# 로지스틱 회귀 실험 노트북

`randomforest.ipynb`에서 만든 체크포인트(`checkpoints/full_gbdt_results.pkl`)를 그대로 불러와서
LightGBM/XGBoost/CatBoost/RandomForest 전체 데이터 OOF 예측을 재사용합니다.
데이터 로드/피처 엔지니어링을 처음부터 다시 하지 않아도 됩니다.

**로지스틱 회귀를 시도하는 이유**: 지금까지의 4개 모델(LGB/XGB/CAT/RF)은 전부 트리 기반이라
"상호작용을 찾는다"는 공통된 귀납적 편향을 가집니다. RandomForest를 추가했을 때도 상관관계가
크게 낮아지지 않았고(0.925~0.931), 개별 성능이 가장 나빠서(-0.23%p) 앙상블에 실제로는
도움이 안 됐습니다(`randomforest.ipynb` 14번 섹션 참고).

로지스틱 회귀는 선형·가법적 모델이라 상호작용을 못 잡는 대신, 트리 계열과 구조적으로
다른 종류의 오차를 만들 가능성이 있습니다. 학습 비용도 GBDT/RF보다 훨씬 저렴해서
(초~분 단위) 20만행 샘플 단계 없이 바로 전체 데이터로 확인합니다.

**이 파일과 `train_ensemble.py`는 같은 폴더에 있어야 아래 import가 동작합니다.**

In [1]:
import sys, os, time
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

sys.path.append(os.getcwd())
from train_ensemble import train_lr

## 1. 체크포인트 로드 — 기존 LGB/XGB/CAT/RF 전체 데이터 결과 재사용

`X_full`, `y_full`, `feat_cols`, `cat_features`는 이미 `build_features`까지 적용된 상태로
저장되어 있으므로 train.csv를 다시 읽거나 피처를 다시 만들 필요가 없습니다.

In [2]:
ckpt = joblib.load("checkpoints/full_gbdt_results.pkl")

X_full, y_full = ckpt["X_full"], ckpt["y_full"]
feat_cols, cat_features = ckpt["feat_cols"], ckpt["cat_features"]

lgb_oof_full = ckpt["lgb_oof_full"]
xgb_oof_full = ckpt["xgb_oof_full"]
cat_oof_full = ckpt["cat_oof_full"]
rf_oof_full = ckpt["rf_oof_full"]

r_full = y_full.mean()
baseline_brier_full = r_full * (1 - r_full)

print(f"복원 완료: X_full{X_full.shape}")
print(f"기준(무정보) Brier = {baseline_brier_full:.5f} (r={r_full:.4f})")

복원 완료: X_full(1475092, 75)
기준(무정보) Brier = 0.24944 (r=0.5238)


## 2. 로지스틱 회귀 학습 (전체 데이터, 147만행)

`train_lr`은 `train_rf`와 동일한 5-fold 구조를 쓰되, 전처리가 다릅니다:
- 수치형: 중앙값 대치 + `StandardScaler` (계수 기반 최적화 수렴 안정성)
- 범주형: 최빈값 대치 + 원-핫 인코딩

GBDT/RF 대비 학습이 훨씬 빠르므로(선형모델 + lbfgs) 20만행 샘플 단계 없이 바로 전체 데이터로 확인합니다.

In [3]:
t0 = time.time()
lr_models_full, lr_oof_full = train_lr(X_full, y_full, cat_features)
print(f"[LogisticRegression] 소요시간: {time.time()-t0:.1f}초")
print(f"[LogisticRegression] OOF Brier (전체): {brier_score_loss(y_full, lr_oof_full):.5f}")
print(f"참고 - 기준 Brier: {baseline_brier_full:.5f}")

  [LR fold 0] brier=0.24612
  [LR fold 1] brier=0.24617
  [LR fold 2] brier=0.24619
  [LR fold 3] brier=0.24617
  [LR fold 4] brier=0.24610
[LogisticRegression] 소요시간: 83.9초
[LogisticRegression] OOF Brier (전체): 0.24615
참고 - 기준 Brier: 0.24944


## 3. 기존 4모델과의 상관관계 확인

핵심 확인 포인트: RF(0.925~0.931)보다 LR이 더 낮은 상관관계를 보이는지.
낮을수록 앙상블 다양성에 유리하지만, 13번 섹션에서 확인했듯 상관관계만으로는
실제 이득을 보장하지 않으므로 4번 섹션에서 숫자로 다시 확인합니다.

In [4]:
oof_corr_full_with_lr = pd.DataFrame({
    "lgb": lgb_oof_full, "xgb": xgb_oof_full, "cat": cat_oof_full,
    "rf": rf_oof_full, "lr": lr_oof_full,
}).corr()
print("LR 포함 상관관계 (전체 데이터 기준, 낮을수록 앙상블에 유리):")
display(oof_corr_full_with_lr)

LR 포함 상관관계 (전체 데이터 기준, 낮을수록 앙상블에 유리):


,lgb,xgb,cat,rf,lr
lgb,1.000000,0.970962,0.964911,0.941018,0.763277
xgb,0.970962,1.000000,0.965874,0.938487,0.754133
cat,0.964911,0.965874,1.000000,0.933647,0.762668
rf,0.941018,0.938487,0.933647,1.000000,0.844365
lr,0.763277,0.754133,0.762668,0.844365,1.000000


## 4. 실제 앙상블 효과 확인 — RF 때와 동일한 방식으로 비교

이전 실험 결과를 기준선으로 둡니다:
- 3모델(LGB/XGB/CAT) 로지스틱 스태킹 Brier: **0.243986**
- 3모델+RF(4모델) 스태킹 Brier: **0.243994** (RF 추가 시 오히려 소폭 악화)

같은 방식으로 RF 대신 LR을 4번째 멤버로 넣었을 때, 그리고 5개 모델을 전부 넣었을 때를
비교해서 LR이 RF보다 실제로 더 도움이 되는지 확인합니다.

In [5]:
iso_lgb_full = IsotonicRegression(out_of_bounds="clip").fit(lgb_oof_full, y_full)
iso_xgb_full = IsotonicRegression(out_of_bounds="clip").fit(xgb_oof_full, y_full)
iso_cat_full = IsotonicRegression(out_of_bounds="clip").fit(cat_oof_full, y_full)
iso_rf_full = IsotonicRegression(out_of_bounds="clip").fit(rf_oof_full, y_full)
iso_lr_full = IsotonicRegression(out_of_bounds="clip").fit(lr_oof_full, y_full)

lgb_c_full = iso_lgb_full.predict(lgb_oof_full)
xgb_c_full = iso_xgb_full.predict(xgb_oof_full)
cat_c_full = iso_cat_full.predict(cat_oof_full)
rf_c_full = iso_rf_full.predict(rf_oof_full)
lr_c_full = iso_lr_full.predict(lr_oof_full)

def stack_brier(components):
    meta_X = np.column_stack(components)
    meta = LogisticRegression().fit(meta_X, y_full)
    pred = meta.predict_proba(meta_X)[:, 1]
    return brier_score_loss(y_full, pred), meta

brier_3, _ = stack_brier([lgb_c_full, xgb_c_full, cat_c_full])
brier_3_rf, _ = stack_brier([lgb_c_full, xgb_c_full, cat_c_full, rf_c_full])
brier_3_lr, meta_3_lr = stack_brier([lgb_c_full, xgb_c_full, cat_c_full, lr_c_full])
brier_5, meta_5 = stack_brier([lgb_c_full, xgb_c_full, cat_c_full, rf_c_full, lr_c_full])

summary_df = pd.DataFrame({
    "조합": ["3모델(LGB/XGB/CAT)", "3모델+RF", "3모델+LR", "3모델+RF+LR(5모델)"],
    "stack_brier": [brier_3, brier_3_rf, brier_3_lr, brier_5],
})
summary_df["vs_3model_pct"] = (brier_3 - summary_df["stack_brier"]) / brier_3 * 100
summary_df["vs_baseline_pct"] = (1 - summary_df["stack_brier"] / baseline_brier_full) * 100
display(summary_df)

print(f"[3모델+LR] 메타 계수: lgb={meta_3_lr.coef_[0][0]:.3f}, xgb={meta_3_lr.coef_[0][1]:.3f}, "
      f"cat={meta_3_lr.coef_[0][2]:.3f}, lr={meta_3_lr.coef_[0][3]:.3f}")
print(f"\n[LR 개별 성능] LR OOF Brier: {brier_score_loss(y_full, lr_oof_full):.5f} "
      f"(참고 - LGB {brier_score_loss(y_full, lgb_oof_full):.5f}, RF {brier_score_loss(y_full, rf_oof_full):.5f})")

,조합,stack_brier,vs_3model_pct,vs_baseline_pct
0,3모델(LGB/XGB/CAT),0.243702,0.000000,2.298574
1,3모델+RF,0.243699,0.000957,2.299508
2,3모델+LR,0.243711,-0.003606,2.295051
3,3모델+RF+LR(5모델),0.243682,0.008160,2.306547


[3모델+LR] 메타 계수: lgb=1.392, xgb=1.345, cat=1.333, lr=0.251

[LR 개별 성능] LR OOF Brier: 0.24615 (참고 - LGB 0.24382, RF 0.24432)


## 5. 체크포인트에 LR 결과 추가 저장

In [6]:
ckpt = joblib.load("checkpoints/full_gbdt_results.pkl")
ckpt["lr_models_full"] = lr_models_full
ckpt["lr_oof_full"] = lr_oof_full
joblib.dump(ckpt, "checkpoints/full_gbdt_results.pkl")
print("LR 결과 추가 저장 완료")

LR 결과 추가 저장 완료
